# ✈️ Aircraft Digital Twin - Core Flow Demo

Notebook này minh họa luồng hoạt động chính yếu của hệ thống, bỏ qua các bước thử nghiệm, test model phụ (như Random Forest) và đi thẳng vào **Digital Twin** với **LSTM** và **PPO**.

In [ ]:
import os
import sys
import numpy as np
import tensorflow as tf

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from scripts.data_processor import prepare_data, FEATURES, KEY_SENSORS
from scripts.aircraft_env import AircraftEnv

## 1. Load Dữ Liệu

In [ ]:
data_dir = os.path.join(PROJECT_ROOT, 'CMAPSSData')
train_rolling, test_rolling, true_rul, scaler = prepare_data(data_dir)
print("Dữ liệu đã được nạp thành công!")

## 2. Load Mô Hình LSTM Đã Lưu

In [ ]:
# Chú ý: Chúng ta không train lại để tiết kiệm thời gian (Do `main.py` đã train và lưu)
model_path = os.path.join(PROJECT_ROOT, 'models', 'lstm_rul_model.keras')

if os.path.exists(model_path):
    lstm_model = tf.keras.models.load_model(model_path)
    print(f"📦 Đã load LSTM model thành công từ: {model_path}")
else:
    print("⚠️ Không tìm thấy file model. Vui lòng chạy `python main.py` một lần để mô hình tự động train và lưu lại!")

## 3. Khởi Tạo Môi Trường Mô Phỏng (RL Environment)

In [ ]:
env = AircraftEnv(
    fleet_data=train_rolling,
    model=lstm_model,
    scaler=scaler,
    sensor_list=KEY_SENSORS,
    features_list=FEATURES
)

print("✈️ Khởi tạo môi trường Digital Twin hoàn tất.")

## 4. Chạy Một Chuyến Bay (Demo Episode)

In [ ]:
obs, info = env.reset()
print("=" * 60)
print(f"🔹 Hành trình bắt đầu!")
print(f"   Động cơ ID: {env.twin.engine_id}")
print(f"   Tổng quãng đường: {env.TOTAL_DISTANCE} | Nhiên liệu đầu vào: {env.twin.fuel}")
print("=" * 60)

done = False
steps = 0
total_reward = 0

while not done and steps < 500:
    # Ở đemo cơ bản này, chúng ta sử dụng random action thay vì PPO action để thấy sự vận hành của Env
    # 0 = Bay tiếp, 1 = Hạ cánh khẩn cấp
    action = 0 if env.twin.current_rul is None or env.twin.current_rul > 40 else 1
    # Đây là logic heuristic đơn giản: Nếu RUL > 40 thì cứ bay, báo động thì hạ cánh ngay (mô phỏng Agent)
    
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    if steps % 20 == 0 or done or 'event' in info:
        action_name = "FLY" if action == 0 else "LAND"
        rul_val = obs[3]
        fuel_val = obs[2]
        dist_val = obs[5]
        print(f"Chu kỳ {steps:>4} | Chọn: {action_name:>4} | RUL: {rul_val:>6.1f} | Xăng: {fuel_val:>5.1f} | Đích: {dist_val:>7.0f}")

print("=" * 60)
print(f"🛬 Kết thúc chuyến bay: {info.get('event', 'Hết thời gian')}")
print(f"   Tổng bước bay: {steps}")
print(f"   Tổng phần thưởng: {total_reward}")